# 04. Data Preparation & Feature Engineering
**Project:** Ethereum Price Prediction Optimization
**Objective:** 
This notebook performs advanced Feature Engineering based on research insights (Lagged Sentiment, Rolling Metrics, Technical Indicators) and prepares the final dataset for modeling.

**Input:** `training_data.csv` (Raw Data)
**Output:** `training_data_4h_engineered.csv` (Processed Data for Modeling)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

%matplotlib inline

# Configuration
DATA_PATH = 'training_data.csv'
OUTPUT_PATH = 'training_data_4h_engineered.csv'


In [2]:
# 1. Load Data
if not os.path.exists(DATA_PATH):
    print(f"Error: {DATA_PATH} not found. Please ensure the raw data file exists.")
else:
    df = pd.read_csv(DATA_PATH)
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime').set_index('datetime')
    print(f"Raw Data Loaded. Shape: {df.shape}")
    print(df.head())


Raw Data Loaded. Shape: (2516, 3)
                           price     volume  Sentiment Score
datetime                                                    
2025-10-07 05:00:00  4711.892090          0              0.0
2025-10-07 06:00:00  4671.623535          0              0.0
2025-10-07 07:00:00  4677.029297  834560000              0.0
2025-10-07 08:00:00  4673.294922  114851840              0.0
2025-10-07 09:00:00  4682.788574  242864128              0.0


In [3]:
# 2. Resampling to 4-Hour Intervals
# Aggregation rules: Price -> Mean, Volume -> Sum, Sentiment -> Mean
df_4h = df.resample('4h').agg({
    'price': 'mean',
    'volume': 'sum', 
    'Sentiment Score': 'mean'
})

# Forward fill price (market doesn't stop, but if gap, assume last price)
df_4h['price'] = df_4h['price'].fillna(method='ffill')
df_4h['Sentiment Score'] = df_4h['Sentiment Score'].fillna(0) # Assume neutral if missing
df_4h.dropna(inplace=True)

print(f"4H Resampled Data Shape: {df_4h.shape}")


4H Resampled Data Shape: (637, 3)


C:\Users\V I C T U S\AppData\Local\Temp\ipykernel_19056\682250553.py:10: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_4h['price'] = df_4h['price'].fillna(method='ffill')


In [4]:
# 3. Feature Engineering (Research Insights)

# A. Lagged Sentiment (Insight: Market reaction is delayed)
for lag in [1, 2, 3, 4, 5, 6]:
    df_4h[f'sent_lag_{lag}'] = df_4h['Sentiment Score'].shift(lag)

# B. Rolling Sentiment (Insight: Single point sentiment is noisy)
df_4h['sent_roll_6'] = df_4h['Sentiment Score'].rolling(window=6).mean()
df_4h['sent_roll_24'] = df_4h['Sentiment Score'].rolling(window=24).mean()

# C. Technical Indicators (Insight: Price momentum drives trends)
# Returns
df_4h['ret_1h'] = df_4h['price'].pct_change(1)
df_4h['ret_4h'] = df_4h['price'].pct_change(4)

# Volatility (Rolling Std Dev)
df_4h['volatility_6h'] = df_4h['price'].rolling(window=6).std()

# RSI (Relative Strength Index)
def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

df_4h['rsi'] = calculate_rsi(df_4h['price'])

# Clean NaN values generated by lags/rolling
df_4h.dropna(inplace=True)
print("Feature Engineering Complete.")
print(df_4h.columns)


Feature Engineering Complete.
Index(['price', 'volume', 'Sentiment Score', 'sent_lag_1', 'sent_lag_2',
       'sent_lag_3', 'sent_lag_4', 'sent_lag_5', 'sent_lag_6', 'sent_roll_6',
       'sent_roll_24', 'ret_1h', 'ret_4h', 'volatility_6h', 'rsi'],
      dtype='object')


In [5]:
# 4. Target Generation (Directional Prediction)
# Insight: Predicting exact price is hard; Direction is more stable.

HORIZON = 4 # Predict 4 steps ahead (4 x 4h = 16h? No, resampled to 4h, so 4 steps = 16h context usually, but here horizon=1 means next 4h candle)
# Actually, 'shift(-1)' means the NEXT 4h candle. 
# Let's predict the NEXT 4H direction.

df_4h['future_price'] = df_4h['price'].shift(-1)
df_4h['target_dir'] = (df_4h['future_price'] > df_4h['price']).astype(int)

# Drop the last row (where future_price is NaN)
df_4h.dropna(inplace=True)

print(f"Target Created. Final Dataset Shape: {df_4h.shape}")
print("Class Balance:")
print(df_4h['target_dir'].value_counts(normalize=True))


Target Created. Final Dataset Shape: (613, 17)
Class Balance:
target_dir
1    0.517129
0    0.482871
Name: proportion, dtype: float64


In [6]:
# 5. Export Processed Data
# This file will be the input for 06_Eth_Price_Prediction_Optimized.ipynb

# Save to CSV
df_4h.to_csv(OUTPUT_PATH)

print(f"SUCCESS: Processed data saved to {OUTPUT_PATH}")
print("You can now proceed to Notebook 06.")


SUCCESS: Processed data saved to training_data_4h_engineered.csv
You can now proceed to Notebook 06.
